In [ ]:
pip install Pillow


In [ ]:
pip install tensorflow


In [ ]:
pip install pandas


In [ ]:
pip install numpy

In [ ]:
pip install matplotlib

In [ ]:
# Link to folder
import os
os.chdir("/home/guest/bmax/imagemodel/dataset")

In [ ]:
!ls

In [ ]:
data = []
label = []

In [ ]:
#Load file
import glob
Happy = glob.glob('Happy/*.jpg') #0
Sad = glob.glob('Sad/*.jpg') #1
Neutral = glob.glob('Neutral/*.jpg') #2

In [ ]:
data = list(data)
label = list(label)

In [ ]:
# import numpy as np
# data1 = np.array(data)

In [ ]:
import tensorflow as tf
import numpy as np
for i in Happy:
  image = tf.keras.preprocessing.image.load_img(i,target_size=(224,224))
  image = np.array(image)
  data.append(image)
  label.append(0)

In [ ]:
def count():
  a = 0
  for i in Sad:
    a =  a + 1
  print (a)
print (count())

In [ ]:
def count():
  a = 0
  for i in Happy:
    a =  a + 1
  print (a)
print (count())

In [ ]:
def count():
  a = 0
  for i in Neutral:
    a =  a + 1
  print (a)
print (count())

In [ ]:
data1 = np.array(data)
label1=np.array(label)
data1.shape

In [ ]:
label1.shape

In [ ]:

import tensorflow as tf
import numpy as np
for i in Neutral:
  image = tf.keras.preprocessing.image.load_img(i,target_size=(224,224))
  image = np.array(image)
  data.append(image)
  label.append(2)

In [ ]:
import tensorflow as tf
import numpy as np
for i in Sad:
  image = tf.keras.preprocessing.image.load_img(i,target_size=(224,224))
  image = np.array(image)
  data.append(image)
  label.append(1)

In [ ]:
#Split data to train
data = np.array(data)
label = np.array(label)
from sklearn.model_selection import train_test_split
X_train, X_test, ytrain, ytest = train_test_split(data, label, test_size=0.1,
                                                 random_state=42)

In [ ]:
data.shape

In [ ]:
X_train.shape

In [ ]:
X_train = X_train / 255.0
X_test  = X_test  / 255.0


In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(3, activation='softmax')(x)

resnet_model = Model(inputs=base_model.input, outputs=output)
resnet_model.compile(optimizer=Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
resnet_model.summary()


In [ ]:
# CELL 3 — Phase 1: Train head only (fast, ~15 epochs)
# ============================================================
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

print("Phase 1: Training classification head (base frozen)...")
history_phase1 = resnet_model.fit(
    X_train, ytrain,
    epochs=15,
    batch_size=32,
    shuffle=True,
    validation_split=0.1,
    verbose=1,
    callbacks=callbacks
)

In [ ]:
# ============================================================
# CELL 4 — Phase 2: Fine-tune last 30 layers of ResNet50
# ============================================================
print("\nPhase 2: Fine-tuning last 30 layers of ResNet50...")

# Unfreeze the whole base, then refreeze everything except last 30 layers
resnet_base.trainable = True
for layer in resnet_base.layers[:-30]:
    layer.trainable = False

# Recompile with very small learning rate to avoid destroying pre-trained weights
resnet_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_phase2 = resnet_model.fit(
    X_train, ytrain,
    epochs=30,
    batch_size=32,
    shuffle=True,
    validation_split=0.1,
    verbose=1,
    callbacks=callbacks
)


In [ ]:
# ============================================================
# CELL 5 — Evaluate & Save
# ============================================================
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

pred_resnet = np.argmax(resnet_model.predict(X_test), axis=1)

# Reuse your existing plot_confusion_matrix function
plot_confusion_matrix(
    confusion_matrix(ytest, pred_resnet),
    classes=['Happy', 'Sad', 'Angry'],
    normalize=True,
    title='ResNet50 Confusion Matrix'
)
plt.show()

test_loss, test_acc = resnet_model.evaluate(X_test, ytest, verbose=0)
print(f"ResNet50 Test Accuracy: {test_acc * 100:.2f}%")

resnet_model.save('resnet50_emotion.keras')
print("Saved: resnet50_emotion.keras")
